In [ ]:
#bing images downloader
import os
import requests
from bs4 import BeautifulSoup
from PIL import Image
from io import BytesIO
from urllib.parse import parse_qs, urlparse
from tqdm import tqdm #progress bar

In [ ]:


def download_bing_images(query, num_images=10, directory='BING_images'):
    base_url = "https://www.bing.com"
    url = f"{base_url}/images/search?q={query.replace(' ', '+')}&form=HDRSC2"
    response = requests.get(url)
    soup = BeautifulSoup(response.text, 'html.parser')
    links = [a['href'] for a in soup.find_all('a', {'class': 'iusc'})]

    if not os.path.exists(directory):
        os.makedirs(directory)

    # Add progress bar
    for i, link in tqdm(enumerate(links[:num_images]), 
                       desc=f'Downloading {query} images', 
                       total=min(len(links), num_images)):
        try:
            parsed_url = urlparse(link)
            image_url = parse_qs(parsed_url.query)['mediaurl'][0]

            img_data = requests.get(image_url).content
            img = Image.open(BytesIO(img_data))
            img.save(os.path.join(directory, f'{query}_{i}.jpg'))
        except Exception as e:
            print("\nskipping undownloadable image")




#MAIN

In [ ]:
download_bing_images('MEOW! pro', num_images=5)

In [ ]:


# once all images downloaded, delete any images that are smaller than 512x512
# Add progress bar

#first, show how many images are in the directory
print(f"Number of images in directory: {len(os.listdir('BING_images'))}")

for filename in tqdm(os.listdir('BING_images'), desc='Processing images'):
    img_path = os.path.join('BING_images', filename)
    try:
        with Image.open(img_path) as img:
            # Get dimensions while image is open
            width, height = img.size
            
            # Check if too small
            if width < 512 or height < 512:
                # Close image before deleting
                img.close()
                os.remove(img_path)
                continue
            
                
    except Exception as e:
        print(f"Error processing {filename}: {e}")

#show how many images are left
print(f"Number of images in directory after deleting images <512px: {len(os.listdir('BING_images'))}")


# now we have a directory of images that are all 512x512 or larger. use BIRME to resize all images
